# LLM graph creation demo

Anton Antonov   
[RakuForPrediction blog at WordPress](https://rakuforprediction.wordpress.com)  
September 2026

----

## Introduction

In this notebook we demonstrate the creation of LLM-graphs provided by the Raku package ["LLM::Graph"](https://raku.land/zef:antononcube/LLM::Graph), [AAp1], by using the [agent skill provided by that package](https://github.com/antononcube/Raku-LLM-Graph/tree/main/resources/raku-llm-graph-making).

----

## Setup

In [1]:
use LLM::Graph;
use LLM::Functions;
use LLM::Tooling;
use LLM::Resources;

use Math::NumberTheory;
use Data::Importers;

In [2]:
my $conf = llm-configuration('ollama', model => 'gemma3:4b')

LLM::Configuration(:name("ollama"), :model("gemma3:4b"), :module("WWW::Ollama"), :max-tokens(8192))

----

## Progressive disclosure LLM-graph

In this section we create the rules of an LLM-graph for "progressive disclosure" ingestion. That graph can be used in an LLM-tool (for external function evaluation) that would allow the appropriate review, selection, and use of agent skills.

The graph below classifies a user command, then evaluates exactly one of three context nodes. The `test-function` on each context node uses the classifier result, so the unrelated context nodes are skipped. The final node is ordinary Raku code and returns a hash with the requested shape.

In [ ]:
my %rules =
    Classification => sub ($command) {
        qq:to/PROMPT/;
        Classify the following user command into exactly one label:
        - skill-info: asks which skills are available, installed, or applicable
        - skill-description: asks what a particular skill does or how to use it
        - code-generation: asks to write, modify, explain, or debug code

        Reply with only one of: skill-info, skill-description, code-generation.

        User command: $command
        PROMPT
    },

    SkillDirectory => { 
        eval-function => sub ($directory = Whatever) { 
            # TODO: Verification of a valid agent skill directory.
            return $directory;
        }
    },

    SkillInfoContext => {
        eval-function => sub ($command, $SkillDirectory) {
            my $text = slurp($SkillDirectory.IO.add('SKILL.md'));
            my %header = do with slurp($SkillDirectory.add('SKILL.md')) ~~ / ^ '-' ** 3..* \v 'name:' (.+?) \s+ 'description:' (.+?) \v '-' ** 3..* / { 
                %(name => $0.Str.trim, description => $1.Str.trim ) 
            }
            return %header;
        },
        test-function => sub ($Classification, $SkillDirectory) {
           $SkillDirectory.defined && $Classification.trim.lc eq 'skill-info'
        }
    },

    SkillDescriptionContext => {
        eval-function => sub ($command, $SkillDirectory) {
            my $text = slurp($SkillDirectory.IO.add('SKILL.md'));
            return $text;
        },
        test-function => sub ($Classification, $SkillDirectory) {
            $SkillDirectory.defined && $Classification.trim.lc eq 'skill-description'
        }
    },

    CodeGenerationContext => {
        eval-function => sub ($command, $SkillDirectory) {
            my $description = slurp($SkillDirectory.IO.add('SKILL.md'));
            my @references;
            if $SkillDirectory.IO.add('reference').d {
               @references = $SkillDirectory.IO.add('reference').dir.map(*.&slurp)
            }
            return {:$description, guide => @references.join("\n\n")}       
        },
        test-function => sub ($Classification, $SkillDirectory) {
            $SkillDirectory.defined && $Classification.trim.lc eq 'code-generation'
        }
    },

    Result => {
        eval-function => sub ($command, $SkillDirectory, $Classification, $SkillInfoContext, $SkillDescriptionContext, $CodeGenerationContext) {
            
            return 'No directory was provided.' 
            unless $SkillDirectory.defined && $SkillDirectory.IO.d;

            my $context = $SkillInfoContext // $SkillDescriptionContext // $CodeGenerationContext;
            {
                command   => $command,
                directory => $SkillDirectory,
                label   => $Classification.trim.lc,
                context => $context
            }
        }
    };

my $graph = llm-graph(%rules, e => $conf):async:progress;

#die $graph.node-spec-errors.join("\n") unless $graph.has-valid-node-specs;
#$graph.create-graph('Describe the raku-llm-graph-making skill.');


LLM::Graph(size => 6, nodes => Classification, CodeGenerationContext, Result, SkillDescriptionContext, SkillDirectory, SkillInfoContext)

The skill directory:

In [18]:
my $directory = $*HOME.add(<.codex skills raku-llm-graph-making>);
say $directory.IO.d;

True


Evaluation:

In [19]:
#my $command = 'Describe the skill.';
my $command = 'I want to generate Raku code for LLM graphs.';
$graph.eval(:$command, :$directory);

Test function awaiting for the results of Classification
Test function awaiting for the results of Classification
Test function awaiting for the results of Classification
Awaiting for the results of Classification


LLM::Graph(size => 6, nodes => Classification, CodeGenerationContext, Result, SkillDescriptionContext, SkillDirectory, SkillInfoContext)

Classification result:

In [20]:
$graph.nodes<Classification><result>.result

code-generation

In [21]:
$graph.nodes<Result><result>».chars

{command => 44, context => {description => 3589, guide => 11582}, directory => 50, label => 15}

Graph plot:

In [23]:
#% html
$graph.dot(theme => 'default', :6graph-size, :8vertex-font-size, vertex-width => 1.2, edge-color => 'SlateGray'):svg

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
 
 Classification_Cluster 
 
 
<!-- Classification -->
 
 Classification 
 
 Classification 
 
<!-- CodeGenerationContext -->
 
 CodeGenerationContext 
 
 CodeGenerationContext 
 
<!-- Classification->CodeGenerationContext -->
 
 Classification->CodeGenerationContext 
 
 
 
<!-- Result -->
 
 Result 
 
 Result 
 
<!-- Classification->Result -->
 
 Classification->Result 
 
 
 
<!-- SkillDescriptionContext -->
 
 SkillDescriptionContext 
 
 SkillDescriptionContext 
 
<!-- Classification->SkillDescriptionContext -->
 
 Classification->SkillDescriptionContext 
 
 
 
<!-- SkillInfoContext -->
 
 SkillInfoContext 
 
 SkillInfoContext 
 
<!-- Classification->SkillInfoContext -->
 
 Classification->SkillInfoContext 
 
 
 
<!-- CodeGenerationContext->Result -->
 
 CodeGenerationContext->Result 
 
 
 
<!-- SkillDescriptionContext->Result -->
 
 SkillDescriptionContext->Result 
 
 
 
<!-- SkillDirectory -->
 
 SkillDirectory 
 
 SkillDirectory 
 
<!-- SkillDirectory->CodeGenerationContext -->
 
 SkillDirectory->CodeGenerationContext 
 
 
 
<!-- SkillDirectory->Result -->
 
 SkillDirectory->Result 
 
 
 
<!-- SkillDirectory->SkillDescriptionContext -->
 
 SkillDirectory->SkillDescriptionContext 
 
 
 
<!-- SkillDirectory->SkillInfoContext -->
 
 SkillDirectory->SkillInfoContext 
 
 
 
<!-- SkillInfoContext->Result -->
 
 SkillInfoContext->Result 
 
 
 
<!-- command -->
 
 command 
 
 command 
 
<!-- command->Classification -->
 
 command->Classification 
 
 
 
<!-- command->CodeGenerationContext -->
 
 command->CodeGenerationContext 
 
 
 
<!-- command->Result -->
 
 command->Result 
 
 
 
<!-- command->SkillDescriptionContext -->
 
 command->SkillDescriptionContext 
 
 
 
<!-- command->SkillInfoContext -->
 
 command->SkillInfoContext 
 
 
 
<!-- directory -->
 
 directory 
 
 directory 
 
<!-- directory->SkillDirectory -->
 
 directory->SkillDirectory

----

## LLM tool creation

---

## Number Theory skill making

Using the exports from the CLI script [`number-theory`](https://raw.githubusercontent.com/antononcube/Raku-Math-NumberTheory/refs/heads/main/bin/number-theory) of ["Math::NumberTheory"](https://raku.land/zef:antononcube/Math::NumberTheory):

In [3]:
my $cli-code=data-import('https://raw.githubusercontent.com/antononcube/Raku-Math-NumberTheory/refs/heads/main/bin/number-theory', 'asis');
text-stats($cli-code)

(chars => 4744 words => 594 lines => 157)

In [4]:
my @sub-names = do with $cli-code ~~ / 'my' \s+ 'constant' \s+ '@exports' (.*?) ';' / { $/.Str.EVAL };
deduce-type(@sub-names)

Vector(Atom((Str)), 58)

In [5]:
my @defs = @sub-names.map({ my $def = try sub-info( ::('&' ~ $_) ); $! ?? Nil !! Pair.new($_, $def) }).grep(* ~~ Pair);
@defs.elems

57

In [8]:
#% html
my @field-names = <name description>;
@defs.map(*.value.grep({ $_.key ∈ @field-names }).Hash).List
==> to-dataset()
==> to-html(:@field-names, align => 'left')

name,description
abundant-number,Compute the nth abundant number
are-coprime,Give true if the arguments are coprime.
chinese-remainder,"Give the smallest x with x>=0 that satisfies all the integer congruences x mod m_i == r_i mod m_i. C<:@r> -- List of remainders. C<:@m> -- List of modules C<:$d> -- Number, lower limit of the result."
continued-fraction,Generate a list of the first $n terms in the continued fraction representation of $x.
convergents,Give a list of the convergents corresponding to the given continued fraction terms list or a number.
cousin-primes,Get the first n-pairs of primes that differ by 4.
deficient-number,Compute the nth deficient number.
digit-count,gives the number of digits in a base representation of the argument. C<$n> -- Integer C<:$base> - Base (integer) C<:$digits> -
divisor-sigma,"Give the divisor function σ(exp, n)."
divisors,Give a list of the integers that divide the argument.
